[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/44_infonce_loss.ipynb)

# 🔴 Hard: InfoNCE Contrastive Loss

Implement a modern **InfoNCE / CLIP-style contrastive loss** with in-batch negatives.

$$
	ext{logits}_{ij} = 
rac{\langle \hat z_i^{(1)}, \hat z_j^{(2)} 
angle}{T}
$$

### Signature
```python
def infonce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    # z1, z2: (B, D) paired embeddings
    # Use in-batch negatives and return a scalar loss
```

### Rules
- Normalize embeddings before computing similarities
- Use the batch index as the positive label
- Average cross-entropy in both directions (`z1 -> z2` and `z2 -> z1`)


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn.functional as F


In [13]:
# ✏️ YOUR IMPLEMENTATION HERE
def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    x_max = x.max(dim=dim, keepdim=True).values
    e_x = torch.exp(x - x_max)
    return e_x / e_x.sum(dim=dim, keepdim=True)

def infonce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    # pass
    z1 = F.normalize(z1, dim=-1) # 【默认】L2归一化后，点积变成了余弦相似度，取值范围严格限制在 $[-1, 1]
    z2 = F.normalize(z2, dim=-1)
    sim = (z1 @ z2.T) / temperature # (b,b)
    z12_loss = torch.diag(-torch.log(my_softmax(sim))) # [range(z1.shape[0]), range(z2.shape[0])]
    z21_loss = torch.diag(-torch.log(my_softmax(sim.T))) #[range(z2.shape[0]), range(z1.shape[0])]
    # return pair_loss.mean()
    return (z12_loss.mean()+z21_loss.mean())/2





In [14]:
# 🧪 Debug
z = torch.eye(4)
print('Aligned loss:', infonce_loss(z, z, temperature=0.07))
print('Shuffled loss:', infonce_loss(z, z[torch.tensor([1, 0, 3, 2])], temperature=0.07))


Aligned loss: tensor(1.8179e-06)
Shuffled loss: tensor(14.2857)


In [15]:
# ✅ SUBMIT
from torch_judge import check
check('infonce_loss')



🧪 Testing: InfoNCE Contrastive Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Scalar output (13.6ms)
  ✅ [2/5] Matches symmetric reference formula (1.2ms)
  ✅ [3/5] Aligned pairs beat shuffled pairs (0.5ms)
  ✅ [4/5] Scale invariant after normalization (0.9ms)
  ✅ [5/5] Gradient flow (1.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (17.8ms total)
  Progress saved. Run status() to see your dashboard.

